In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt



In [ ]:
# Function to create LSTM model
def create_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    return model

# Function to predict and calculate MSE for a given target variable
def predict_and_evaluate(grouped_data, target_column):
    mse_results = {}
    prediction_results = []

    for cell, group in grouped_data:
        if len(group) < 4:  # Minimum required for a 60-20-20 split
            continue
        
        # Prepare data
        X = group[['AVG_NO_USER', 'DL_TRAFFIC_MB']].values
        y = group[target_column].values
        
        # Reshape X for LSTM [samples, time steps, features]
        X = X.reshape((X.shape[0], 1, X.shape[1]))
        
        # Split into train, validation, and test sets
        X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
        
        # Create and train the model
        model = create_lstm_model((X_train.shape[1], X_train.shape[2]))
        model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val), verbose=0)
        
        # Predict and calculate MSE
        y_pred = model.predict(X_test)
        train_mse = mean_squared_error(y_train, model.predict(X_train))
        val_mse = mean_squared_error(y_val, model.predict(X_val))
        test_mse = mean_squared_error(y_test, y_pred)
        
        # Store MSE results
        mse_results[cell] = {'train_mse': train_mse, 'val_mse': val_mse, 'test_mse': test_mse}
        
        # Store prediction results for degradation/increase analysis
        prediction_results.append(pd.DataFrame({
            'EUTRANCELLFDD': cell,
            'Actual': y_test.flatten(),
            'Predicted': y_pred.flatten()
        }))

    # Convert MSE results to DataFrame
    mse_df = pd.DataFrame.from_dict(mse_results, orient='index')
    
    # Combine all predictions into a single DataFrame
    predictions_df = pd.concat(prediction_results)
    
    return mse_df, predictions_df

# Function to calculate degradation/increase counts
def calculate_degradation_increase(predictions_df):
    # Calculate the percentage change
    predictions_df['PCT_CHANGE'] = predictions_df.groupby('EUTRANCELLFDD')['Predicted'].pct_change() * 100
    
    # Define conditions for degradation and increase
    degraded_condition = predictions_df['PCT_CHANGE'] <= -50
    increased_condition = predictions_df['PCT_CHANGE'] >= 0.1
    
    # Count the number of cells that meet each condition
    degraded_cells_count = predictions_df[degraded_condition].groupby('EUTRANCELLFDD').ngroups
    increased_cells_count = predictions_df[increased_condition].groupby('EUTRANCELLFDD').ngroups
    
    return degraded_cells_count, increased_cells_count

# Load the dataset
file_path = '../../data/Data3.csv'  # Make sure the file is in the same directory as the code
data = pd.read_csv(file_path)

# Select relevant columns and normalize
columns_to_normalize = ['AVG_NO_USER', 'AVG_USR_THRPUT_DL', 'DL_TRAFFIC_MB']
data = data[['EUTRANCELLFDD'] + columns_to_normalize]

scaler = MinMaxScaler(feature_range=(0, 10))
data[columns_to_normalize] = scaler.fit_transform(data[columns_to_normalize])

# Group data by 'EUTRANCELLFDD'
grouped_data = data.groupby('EUTRANCELLFDD')

# Perform prediction and evaluation for each target variable
mse_thrput_dl, predictions_thrput_dl = predict_and_evaluate(grouped_data, 'AVG_USR_THRPUT_DL')
mse_no_user, predictions_no_user = predict_and_evaluate(grouped_data, 'AVG_NO_USER')
mse_traffic_mb, predictions_traffic_mb = predict_and_evaluate(grouped_data, 'DL_TRAFFIC_MB')

# Compare MSE
print("Comparison of MSE:")
print("AVG_USR_THRPUT_DL:", mse_thrput_dl['test_mse'].mean())
print("AVG_NO_USER:", mse_no_user['test_mse'].mean())
print("DL_TRAFFIC_MB:", mse_traffic_mb['test_mse'].mean())

# Calculate and compare degradation/increase counts
degraded_thrput_dl, increased_thrput_dl = calculate_degradation_increase(predictions_thrput_dl)
degraded_no_user, increased_no_user = calculate_degradation_increase(predictions_no_user)
degraded_traffic_mb, increased_traffic_mb = calculate_degradation_increase(predictions_traffic_mb)

print("\nComparison of Degraded/Improved Cells:")
print(f"AVG_USR_THRPUT_DL: {degraded_thrput_dl} degraded, {increased_thrput_dl} increased")
print(f"AVG_NO_USER: {degraded_no_user} degraded, {increased_no_user} increased")
print(f"DL_TRAFFIC_MB: {degraded_traffic_mb} degraded, {increased_traffic_mb} increased")

3/3 [==============================] - 0s 0s/step


3/3 [==============================] - 1s 2ms/step


7/7 [==============================] - 0s 2ms/step


1/1 [==============================] - 0s 23ms/step


1/1 [==============================] - 1s 1s/step


...

3/3 [==============================] - 1s 3ms/step


7/7 [==============================] - 0s 3ms/step


3/3 [==============================] - 0s 2ms/step


1/1 [==============================] - 1s 948ms/step


3/3 [==============================] - 0s 2ms/step


In [ ]:

c 